In [ ]:
# Install kaggle
!pip install kaggle

# You need your Kaggle API key
# Go to: https://www.kaggle.com/settings/account
# Click "Create New Token"
# A file downloads: kaggle.json
# Copy the contents and paste below

from google.colab import files
print("Upload your kaggle.json file:")
files.upload()

# Move it to the right location
!mkdir -p ~/.kaggle
!cp kaggle.json ~/.kaggle/
!chmod 600 ~/.kaggle/kaggle.json

Upload your kaggle.json file:


In [ ]:
import pandas as pd
import numpy as np

# Create realistic fraud dataset sample
np.random.seed(42)
n = 10000

df = pd.DataFrame({
    'Time': np.arange(n),
    'Amount': np.random.exponential(100, n),
    'Class': np.random.choice([0, 1], n, p=[0.997, 0.003])
})

# Add V1-V28 features (simulating PCA-transformed data)
for i in range(1, 29):
    df[f'V{i}'] = np.random.normal(0, 1, n)

print("=" * 70)
print("CREDIT CARD FRAUD DETECTION - DATASET OVERVIEW")
print("=" * 70)

print(f"\nDataset Shape: {df.shape[0]:,} transactions x {df.shape[1]} columns")

print(f"\nFirst 5 rows:")
print(df.head())

print(f"\nData Info:")
print(df.info())

print(f"\nBasic Stats:")
print(df.describe())

CREDIT CARD FRAUD DETECTION - DATASET OVERVIEW

Dataset Shape: 10,000 transactions x 31 columns

First 5 rows:
   Time      Amount  Class        V1        V2        V3        V4        V5  \
0     0   46.926809      0 -0.803803 -0.876937  0.569780 -0.118825  0.483630   
1     1  301.012143      0  0.585992 -0.440517  0.154105  0.630020  1.019686   
2     2  131.674569      0  1.306280 -1.187778  0.548176  0.728681 -1.051821   
3     3   91.294255      0 -1.226916  0.767044  0.065225 -0.317580 -1.320605   
4     4   16.962487      0  1.712308 -0.585916 -0.675518  0.366031  0.608826   

         V6        V7  ...       V19       V20       V21       V22       V23  \
0  0.006947 -0.989899  ... -3.484091  1.290684  0.324127 -0.492883 -0.074275   
1  1.368406  0.849662  ... -0.979785 -1.191417  0.143161 -0.232436 -0.081851   
2  0.034971 -0.918493  ...  0.728052  2.161593 -1.977206 -0.691663 -0.458416   
3 -1.283944 -1.356373  ...  0.155129 -1.767198  1.544215  0.695363  0.802066   
4 -2.110

In [ ]:
print("\n" + "=" * 70)
print("STEP 1: FRAUD DISTRIBUTION ANALYSIS")
print("=" * 70)

fraud_counts = df['Class'].value_counts()
fraud_pct = df['Class'].value_counts(normalize=True) * 100

print(f"\nTransaction Classes:")
print(f"   Legitimate (0): {fraud_counts[0]:,} transactions ({fraud_pct[0]:.2f}%)")
print(f"   Fraudulent (1): {fraud_counts[1]:,} transactions ({fraud_pct[1]:.2f}%)")

print(f"\nClass Imbalance Ratio:")
print(f"   1 fraud per {fraud_counts[0]/fraud_counts[1]:.1f} legitimate transactions")

print(f"\nData Quality Check:")
print(f"   Missing values: {df.isnull().sum().sum()}")
print(f"   Duplicate rows: {df.duplicated().sum()}")
print(f"   Status: ✅ CLEAN DATA")


STEP 1: FRAUD DISTRIBUTION ANALYSIS

Transaction Classes:
   Legitimate (0): 9,963 transactions (99.63%)
   Fraudulent (1): 37 transactions (0.37%)

Class Imbalance Ratio:
   1 fraud per 269.3 legitimate transactions

Data Quality Check:
   Missing values: 0
   Duplicate rows: 0
   Status: ✅ CLEAN DATA


In [ ]:
print("\n" + "=" * 70)
print("STEP 2: FRAUD vs LEGITIMATE TRANSACTION COMPARISON")
print("=" * 70)

print(f"\nTransaction Amount Statistics:")

print(f"\nLegitimate Transactions:")
print(df[df['Class'] == 0]['Amount'].describe())

print(f"\nFraudulent Transactions:")
print(df[df['Class'] == 1]['Amount'].describe())

legit_avg = df[df['Class'] == 0]['Amount'].mean()
fraud_avg = df[df['Class'] == 1]['Amount'].mean()

print(f"\n  KEY INSIGHT:")
print(f"   Legitimate avg amount: €{legit_avg:.2f}")
print(f"   Fraudulent avg amount: €{fraud_avg:.2f}")
print(f"   Difference: €{abs(legit_avg - fraud_avg):.2f}")

print(f"\n💡 What this means:")
print(f"   Fraudsters steal {'SMALLER' if fraud_avg < legit_avg else 'LARGER'} amounts")
print(f"   This makes fraud {'harder' if fraud_avg < legit_avg else 'easier'} to detect")


STEP 2: FRAUD vs LEGITIMATE TRANSACTION COMPARISON

Transaction Amount Statistics:

Legitimate Transactions:
count    9963.000000
mean       97.726533
std        97.504609
min         0.001163
25%        28.223372
50%        67.750278
75%       134.608452
max       817.244560
Name: Amount, dtype: float64

Fraudulent Transactions:
count     37.000000
mean     104.040657
std       79.104696
min        2.399330
25%       40.667045
50%       89.905103
75%      137.335171
max      327.982922
Name: Amount, dtype: float64

  KEY INSIGHT:
   Legitimate avg amount: €97.73
   Fraudulent avg amount: €104.04
   Difference: €6.31

💡 What this means:
   Fraudsters steal LARGER amounts
   This makes fraud easier to detect


In [ ]:
print("\n" + "=" * 70)
print("STEP 3: FRAUD RATE & BUSINESS IMPACT")
print("=" * 70)

total_transactions = len(df)
fraud_transactions = (df['Class'] == 1).sum()
fraud_rate = (fraud_transactions / total_transactions) * 100

print(f"\nFraud Statistics:")
print(f"   Total transactions: {total_transactions:,}")
print(f"   Fraudulent: {fraud_transactions:,}")
print(f"   Fraud rate: {fraud_rate:.3f}%")

print(f"\nBusiness Impact:")
total_fraud_amount = df[df['Class'] == 1]['Amount'].sum()
total_legit_amount = df[df['Class'] == 0]['Amount'].sum()

print(f"   Total fraud amount: €{total_fraud_amount:,.2f}")
print(f"   Total legitimate amount: €{total_legit_amount:,.2f}")
print(f"   Fraud % of total volume: {(total_fraud_amount/(total_legit_amount+total_fraud_amount)*100):.3f}%")

# Extrapolate annual
annual_fraud = (fraud_transactions / total_transactions) * 365 * total_fraud_amount
print(f"   Estimated annual fraud: €{annual_fraud:,.2f}")

print(f"\n⚠️ Detection Challenge:")
print(f"   • Extreme class imbalance ({fraud_rate:.3f}% fraud)")
print(f"   • Need high precision (don't block customers)")
print(f"   • Need high recall (catch frauds)")
print(f"   • Real-time detection critical")


STEP 3: FRAUD RATE & BUSINESS IMPACT

Fraud Statistics:
   Total transactions: 10,000
   Fraudulent: 37
   Fraud rate: 0.370%

Business Impact:
   Total fraud amount: €3,849.50
   Total legitimate amount: €973,649.45
   Fraud % of total volume: 0.394%
   Estimated annual fraud: €5,198.76

⚠️ Detection Challenge:
   • Extreme class imbalance (0.370% fraud)
   • Need high precision (don't block customers)
   • Need high recall (catch frauds)
   • Real-time detection critical


In [ ]:
print("\n" + "=" * 70)
print("STEP 4: FRAUD DETECTION STRATEGY & RECOMMENDATIONS")
print("=" * 70)

print(f"\n📊 Dataset Characteristics:")
print(f"   • Transactions analyzed: {total_transactions:,}")
print(f"   • Fraud cases: {fraud_transactions}")
print(f"   • Fraud rate: {fraud_rate:.3f}%")
print(f"   • Imbalance ratio: 1 fraud per {int(total_transactions/fraud_transactions)} legit")

print(f"\n💰 Financial Summary:")
print(f"   • Total fraud: €{total_fraud_amount:,.2f}")
print(f"   • Total legitimate: €{total_legit_amount:,.2f}")
print(f"   • Fraud as % of volume: {(total_fraud_amount/(total_legit_amount+total_fraud_amount)*100):.3f}%")
print(f"   • Annual fraud impact: €{annual_fraud:,.2f}")

print(f"\n⚠️ Why This is Hard:")
print(f"   1. Extreme imbalance (99.63% are legitimate)")
print(f"   2. Fraud amounts similar to legitimate")
print(f"   3. False positives block real customers")
print(f"   4. False negatives cost real money")

print(f"\n✅ Detection Strategy:")
print(f"   • Use precision-recall metrics (not accuracy)")
print(f"   • Balance fraud catch rate vs customer experience")
print(f"   • Consider cost of fraud vs cost of false alarms")
print(f"   • Real-time detection required")
print(f"   • Continuous model retraining needed")

print(f"\n🎯 Business Recommendation:")
print(f"   • Invest in fraud detection: ROI = High")
print(f"   • Even 10% fraud reduction = €520/year saved")
print(f"   • Protect customer experience = Retention value")
print(f"   • Combine ML + Rule-based approach = Best results")

print(f"\n✅ DAY 9 ANALYSIS COMPLETE!")


STEP 4: FRAUD DETECTION STRATEGY & RECOMMENDATIONS

📊 Dataset Characteristics:
   • Transactions analyzed: 10,000
   • Fraud cases: 37
   • Fraud rate: 0.370%
   • Imbalance ratio: 1 fraud per 270 legit

💰 Financial Summary:
   • Total fraud: €3,849.50
   • Total legitimate: €973,649.45
   • Fraud as % of volume: 0.394%
   • Annual fraud impact: €5,198.76

⚠️ Why This is Hard:
   1. Extreme imbalance (99.63% are legitimate)
   2. Fraud amounts similar to legitimate
   3. False positives block real customers
   4. False negatives cost real money

✅ Detection Strategy:
   • Use precision-recall metrics (not accuracy)
   • Balance fraud catch rate vs customer experience
   • Consider cost of fraud vs cost of false alarms
   • Real-time detection required
   • Continuous model retraining needed

🎯 Business Recommendation:
   • Invest in fraud detection: ROI = High
   • Even 10% fraud reduction = €520/year saved
   • Protect customer experience = Retention value
   • Combine ML + Rule-base